# WAXAL ASR - Whisper Baseline
Lingala, Shona, Luganda | Metric: 0.5*WER + 0.5*CER

In [ ]:
!pip install -q torch torchaudio datasets transformers accelerate evaluate jiwer soundfile librosa pandas numpy tqdm
import sys
if 'google.colab' in sys.modules:
    !apt-get -qq install ffmpeg
print('OK')

In [ ]:
import os, random, gc
import numpy as np
import torch
import pandas as pd
import librosa
from datasets import load_dataset, Dataset, interleave_datasets
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
)
from evaluate import load as load_metric
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Mount Google Drive to save/load model checkpoints
from google.colab import drive
drive.mount('/content/drive')
import os
MODEL_PATH = '/content/drive/MyDrive/waxal_whisper_base'
SKIP_TRAINING = os.path.exists(f'{MODEL_PATH}/config.json')
print(f'Model in Drive: {SKIP_TRAINING}')


## 1. Load WAXAL (streaming = no disk issues)

In [ ]:
target_langs = ['lin', 'sna', 'lug']
configs = [f'{lang}_asr' for lang in target_langs]
print(f'Loading: {configs}')

all_train, all_val, all_test = [], [], []
for config in configs:
    print(f'  {config}...')
    ds = load_dataset('google/WaxalNLP', config, streaming=True)
    all_train.append(ds['train'])
    all_val.append(ds['validation'])
    all_test.append(ds['test'])

train_ds = interleave_datasets(all_train)
val_ds = interleave_datasets(all_val)
test_ds = interleave_datasets(all_test)

NUM_TRAIN = 80
NUM_VAL = 10
train_iter = train_ds.shuffle(seed=42).take(NUM_TRAIN)
val_iter = val_ds.shuffle(seed=42).take(NUM_VAL)
print(f'Train: {NUM_TRAIN}, Val: {NUM_VAL}')

## 2. Load Whisper

In [ ]:
model_name = 'openai/whisper-base'
# Always load processor/tokenizer (needed for inference too)
processor = WhisperProcessor.from_pretrained(model_name, language='en', task='transcribe')
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
tokenizer = WhisperTokenizer.from_pretrained(model_name, language='en', task='transcribe')

if not SKIP_TRAINING:
    model = WhisperForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    print(f'Training model params: {model.num_parameters():,}')
else:
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
    print('Loaded model from Drive')


## 3. Preprocess (gc between steps to save RAM)

In [ ]:
def prepare_example(ex):
    audio = ex['audio']
    arr = audio['array']
    sr = audio['sampling_rate']
    if sr != 16000:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    feat = feature_extractor(arr, sampling_rate=16000).input_features[0]
    labels = tokenizer(ex['transcription']).input_ids
    return {'input_features': feat, 'labels': labels}

train_list = []
for ex in train_iter:
    train_list.append(prepare_example(ex))
    del ex
gc.collect()

val_list = []
for ex in val_iter:
    val_list.append(prepare_example(ex))
    del ex
gc.collect()

train_dataset = Dataset.from_list(train_list)
val_dataset = Dataset.from_list(val_list)
del train_list, val_list
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
def data_collator(features):
    feat = torch.tensor(np.array([f['input_features'] for f in features]), dtype=torch.float)
    labels = [torch.tensor(f['labels']) for f in features]
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    return {'input_features': feat, 'labels': labels}

wer_metric = load_metric('wer')
cer_metric = load_metric('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    w = wer_metric.compute(predictions=pred_str, references=label_str)
    c = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': w, 'cer': c, 'combined': 0.5 * w + 0.5 * c}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-waxal',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=10,
    num_train_epochs=3,
    eval_strategy='steps',
    eval_steps=25,
    save_steps=50,
    logging_steps=5,
    report_to=['none'],
    predict_with_generate=True,
    generation_max_length=225,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    seed=42,
    data_seed=42,
    load_best_model_at_end=True,
    metric_for_best_model='combined',
    greater_is_better=False,
)

In [ ]:
if not SKIP_TRAINING:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(MODEL_PATH)
    print('Model saved to Drive')
else:
    print('Skipping training - loading model from Drive...')
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)


## 4. Generate Submission

In [ ]:
model.to(device)
model.eval()

predictions = []
batch = []
for i, ex in enumerate(test_ds):
    batch.append(ex)
    if len(batch) == 16:
        inputs, ids = collate_test(batch)
        with torch.no_grad():
            generated = model.generate(inputs)
        transcriptions = processor.batch_decode(generated, skip_special_tokens=True)
        for idx, trans in zip(ids, transcriptions):
            predictions.append({'id': idx, 'transcription': trans})
        batch = []
        if (i + 1) % 160 == 0:
            print(f'  {i+1} done')
if batch:
    inputs, ids = collate_test(batch)
    with torch.no_grad():
        generated = best_model.generate(inputs)
    transcriptions = processor.batch_decode(generated, skip_special_tokens=True)
    for idx, trans in zip(ids, transcriptions):
        predictions.append({'id': idx, 'transcription': trans})

submission_df = pd.DataFrame(predictions)
submission_df.to_csv('baseline_whisper_submission.csv', index=False)
print(f'Saved {len(submission_df)} predictions')
